References
- [How sqlite-vec Works for Storing and Querying Vector Embeddings](https://medium.com/@stephenc211/how-sqlite-vec-works-for-storing-and-querying-vector-embeddings-165adeeeceea)
- [Using sqlite-vec in Python](https://alexgarcia.xyz/sqlite-vec/python.html)

In [98]:
import sqlite3
import sqlite_vec
import json
import struct
from sqlite_vec import serialize_float32

import numpy as np
import pandas as pd

DB = 'embedding.db'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [99]:
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    CREATE VIRTUAL TABLE IF NOT EXISTS content USING vec0(
        id INTEGER PRIMARY KEY,
        embedding float[32]);
    '''
    conn.execute(sql)


In [100]:
embedding = np.random.rand(32)
embedding

array([0.17928646, 0.28678521, 0.40561139, 0.42518374, 0.57131171,
       0.78538709, 0.90533793, 0.22019145, 0.75054853, 0.92835748,
       0.22822954, 0.28514233, 0.1881725 , 0.98189099, 0.77661858,
       0.01711322, 0.69259698, 0.34281377, 0.57125174, 0.21630612,
       0.83298981, 0.31594113, 0.56210189, 0.84398574, 0.00527638,
       0.53463228, 0.01233109, 0.98375881, 0.27609285, 0.73126858,
       0.360263  , 0.71403649])

In [101]:
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = 'INSERT OR IGNORE INTO content (id, embedding) VALUES (?, ?)'
    conn.execute(sql, (1, serialize_float32(embedding.astype(np.float32))))

In [102]:
data = [(2 + i, v.astype(np.float32)) for i, v in enumerate(np.random.rand(9, 32))]

with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = 'INSERT OR IGNORE INTO content (id, embedding) VALUES (?, ?)'
    conn.executemany(sql, data)

In [103]:
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    SELECT id, 
        vec_to_json(embedding) AS embedding
    FROM content'''
    df = pd.read_sql(sql, conn)
df['embedding'] = df['embedding'].map(json.loads)
df

,id,embedding
0,1,"[0.179286, 0.286785, 0.405611, 0.425184, 0.571..."
1,2,"[0.784749, 0.580171, 0.719782, 0.682191, 0.048..."
2,3,"[0.212973, 0.13172, 0.292978, 0.842693, 0.0874..."
3,4,"[0.298993, 0.05479, 0.589087, 0.093957, 0.6666..."
4,5,"[0.625578, 0.752711, 0.357756, 0.338014, 0.247..."
5,6,"[0.450362, 0.963078, 0.37736, 0.388481, 0.3135..."
6,7,"[0.412468, 0.945932, 0.414899, 0.12436, 0.6607..."
7,8,"[0.674546, 0.949668, 0.375858, 0.128354, 0.243..."
8,9,"[0.195239, 0.991116, 0.879207, 0.260038, 0.504..."
9,10,"[0.100905, 0.541015, 0.553886, 0.519489, 0.666..."


In [104]:
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    SELECT
      id,
      distance
    FROM content
    WHERE embedding match '[0.890, 0.544, 0.825, 0.961, 0.358, 0.0196, 0.521, 0.175, 0.890, 0.544, 0.825, 0.961, 0.358, 0.0196, 0.521, 0.175, 0.890, 0.544, 0.825, 0.961, 0.358, 0.0196, 0.521, 0.175, 0.890, 0.544, 0.825, 0.961, 0.358, 0.0196, 0.521, 0.175]'
    ORDER BY distance
    LIMIT 2;
    '''
    df = pd.read_sql(sql, conn)
df

,id,distance
0,5,1.984164
1,2,2.130458


In [105]:
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    SELECT
      id,
      distance
    FROM content
    WHERE embedding match ?
    ORDER BY distance
    LIMIT 2;
    '''
    df = pd.read_sql(sql, conn, params=(json.dumps(list(embedding.astype(float))), ))
df

,id,distance
0,1,0.000000
1,4,1.843627


In [106]:
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    SELECT
      id,
      distance
    FROM content
    WHERE embedding match ?
    ORDER BY distance
    LIMIT 2;
    '''
    df = pd.read_sql(sql, conn, params=(serialize_float32(embedding.astype(float)), ))
df

,id,distance
0,1,0.000000
1,4,1.843627


In [115]:
v = serialize_float32(embedding.astype(float))
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    SELECT
      id,
      vec_to_json(embedding),
      vec_to_json(vec_add(embedding, ?)) AS addition,
      vec_to_json(vec_normalize(embedding)),
      vec_distance_l2(embedding, ?),
      vec_distance_l1(embedding, ?),
      vec_distance_cosine(embedding, ?),
    FROM content
    '''
    df = pd.read_sql(sql, conn, params=(v, v, v, v))
df

,id,vec_to_json(embedding),addition,vec_to_json(vec_normalize(embedding)),"vec_distance_l2(embedding, ?)","vec_distance_l1(embedding, ?)","vec_distance_cosine(embedding, ?)"
0,1,"[0.179286,0.286785,0.405611,0.425184,0.571312,...","[0.358573,0.573570,0.811223,0.850367,1.142623,...","[0.054735,0.087553,0.123830,0.129805,0.174417,...",0.000000,0.000000,1.110223e-16
1,2,"[0.784749,0.580171,0.719782,0.682191,0.048266,...","[0.964035,0.866956,1.125393,1.107375,0.619578,...","[0.233872,0.172904,0.214511,0.203308,0.014384,...",2.497737,11.804899,2.835189e-01
2,3,"[0.212973,0.131720,0.292978,0.842693,0.087436,...","[0.392259,0.418505,0.698590,1.267877,0.658748,...","[0.074622,0.046153,0.102655,0.295267,0.030636,...",2.376339,9.989146,2.925236e-01
3,4,"[0.298993,0.054790,0.589087,0.093957,0.666678,...","[0.478279,0.341576,0.994698,0.519140,1.237990,...","[0.092320,0.016918,0.181891,0.029011,0.205849,...",1.843627,8.361620,1.601363e-01
4,5,"[0.625578,0.752711,0.357756,0.338014,0.247918,...","[0.804865,1.039496,0.763367,0.763198,0.819229,...","[0.217882,0.262161,0.124603,0.117727,0.086347,...",2.225263,9.562289,2.545683e-01
5,6,"[0.450362,0.963078,0.377360,0.388481,0.313576,...","[0.629649,1.249863,0.782971,0.813665,0.884887,...","[0.144849,0.309754,0.121370,0.124947,0.100855,...",1.992643,9.151027,1.935799e-01
6,7,"[0.412468,0.945932,0.414899,0.124360,0.660726,...","[0.591754,1.232717,0.820511,0.549544,1.232038,...","[0.126194,0.289407,0.126938,0.038048,0.202149,...",2.455028,10.972016,2.814781e-01
7,8,"[0.674546,0.949668,0.375858,0.128354,0.243438,...","[0.853832,1.236453,0.781470,0.553538,0.814750,...","[0.222173,0.312789,0.123795,0.042276,0.080181,...",1.946806,8.480429,1.876687e-01
8,9,"[0.195239,0.991116,0.879207,0.260038,0.504475,...","[0.374525,1.277901,1.284818,0.685222,1.075787,...","[0.055771,0.283116,0.251149,0.074281,0.144105,...",2.431159,11.501503,2.555114e-01
9,10,"[0.100905,0.541015,0.553886,0.519489,0.666759,...","[0.280192,0.827800,0.959497,0.944673,1.238071,...","[0.031122,0.166861,0.170831,0.160222,0.205644,...",1.925653,8.881992,1.745252e-01


In [120]:
v = serialize_float32(embedding.astype(float))
with sqlite3.connect(DB) as conn:
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    sql = '''
    SELECT
      id,
      vec_to_json(embedding),
      vec_to_json(vec_add(embedding, ?)) AS addition,
      vec_to_json(vec_normalize(embedding)),
      vec_to_json(vec_quantize_int8(embedding, 'unit')),
      vec_to_json(vec_quantize_binary(embedding))
    FROM content
    '''
    df = pd.read_sql(sql, conn, params=(v, ))
df

,id,vec_to_json(embedding),addition,vec_to_json(vec_normalize(embedding)),"vec_to_json(vec_quantize_int8(embedding, 'unit'))",vec_to_json(vec_quantize_binary(embedding))
0,1,"[0.179286,0.286785,0.405611,0.425184,0.571312,...","[0.358573,0.573570,0.811223,0.850367,1.142623,...","[0.054735,0.087553,0.123830,0.129805,0.174417,...","[22,36,51,53,72,99,114,27,95,117,28,35,23,124,...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
1,2,"[0.784749,0.580171,0.719782,0.682191,0.048266,...","[0.964035,0.866956,1.125393,1.107375,0.619578,...","[0.233872,0.172904,0.214511,0.203308,0.014384,...","[99,73,91,86,5,16,9,73,113,112,23,29,104,114,3...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
2,3,"[0.212973,0.131720,0.292978,0.842693,0.087436,...","[0.392259,0.418505,0.698590,1.267877,0.658748,...","[0.074622,0.046153,0.102655,0.295267,0.030636,...","[26,16,36,106,10,98,2,23,26,13,65,13,36,126,12...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
3,4,"[0.298993,0.054790,0.589087,0.093957,0.666678,...","[0.478279,0.341576,0.994698,0.519140,1.237990,...","[0.092320,0.016918,0.181891,0.029011,0.205849,...","[37,6,74,11,84,43,103,15,126,124,112,45,45,36,...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
4,5,"[0.625578,0.752711,0.357756,0.338014,0.247918,...","[0.804865,1.039496,0.763367,0.763198,0.819229,...","[0.217882,0.262161,0.124603,0.117727,0.086347,...","[79,95,45,42,31,27,97,40,67,108,72,60,21,2,98,...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
5,6,"[0.450362,0.963078,0.377360,0.388481,0.313576,...","[0.629649,1.249863,0.782971,0.813665,0.884887,...","[0.144849,0.309754,0.121370,0.124947,0.100855,...","[56,122,47,49,39,76,7,24,61,38,70,78,2,58,93,2...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
6,7,"[0.412468,0.945932,0.414899,0.124360,0.660726,...","[0.591754,1.232717,0.820511,0.549544,1.232038,...","[0.126194,0.289407,0.126938,0.038048,0.202149,...","[52,120,52,15,83,42,122,118,99,44,5,67,46,6,54...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
7,8,"[0.674546,0.949668,0.375858,0.128354,0.243438,...","[0.853832,1.236453,0.781470,0.553538,0.814750,...","[0.222173,0.312789,0.123795,0.042276,0.080181,...","[85,120,47,15,30,91,113,33,42,88,41,32,24,74,2...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
8,9,"[0.195239,0.991116,0.879207,0.260038,0.504475,...","[0.374525,1.277901,1.284818,0.685222,1.075787,...","[0.055771,0.283116,0.251149,0.074281,0.144105,...","[24,125,111,32,63,75,77,72,70,123,83,68,112,24...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
9,10,"[0.100905,0.541015,0.553886,0.519489,0.666759,...","[0.280192,0.827800,0.959497,0.944673,1.238071,...","[0.031122,0.166861,0.170831,0.160222,0.205644,...","[12,68,70,65,84,43,91,121,87,68,29,27,101,91,2...","[1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1..."
